In [173]:
from random import shuffle

import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [175]:
# read Rainfall.csv and train_prev.csv

df_original = pd.read_csv('data/Rainfall.csv')
df_train = pd.read_csv('data/train_prev.csv')

In [176]:
df_original.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
0,1,1025.9,19.9,18.3,16.8,13.1,72,49,yes,9.3,80.0,26.3
1,2,1022.0,21.7,18.9,17.2,15.6,81,83,yes,0.6,50.0,15.3
2,3,1019.7,20.3,19.3,18.0,18.4,95,91,yes,0.0,40.0,14.2
3,4,1018.9,22.3,20.6,19.1,18.8,90,88,yes,1.0,50.0,16.9
4,5,1015.9,21.3,20.7,20.2,19.9,95,81,yes,0.0,40.0,13.7


In [177]:
# transform rainfall column to binary

df_original['rainfall'] = df_original['rainfall'].map({'no': 0, 'yes': 1})
df_original.columns = df_original.columns.str.strip()
df_original.head()

,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,rainfall,sunshine,winddirection,windspeed
0,1,1025.9,19.9,18.3,16.8,13.1,72,49,1,9.3,80.0,26.3
1,2,1022.0,21.7,18.9,17.2,15.6,81,83,1,0.6,50.0,15.3
2,3,1019.7,20.3,19.3,18.0,18.4,95,91,1,0.0,40.0,14.2
3,4,1018.9,22.3,20.6,19.1,18.8,90,88,1,1.0,50.0,16.9
4,5,1015.9,21.3,20.7,20.2,19.9,95,81,1,0.0,40.0,13.7


In [178]:
df_original.columns

Index(['day', 'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint',
       'humidity', 'cloud', 'rainfall', 'sunshine', 'winddirection',
       'windspeed'],
      dtype='object')

In [179]:
# change column order
new_order = ['pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed', 'rainfall', 'day']
df_original = df_original[new_order]
df_original.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall,day
0,1025.9,19.9,18.3,16.8,13.1,72,49,9.3,80.0,26.3,1,1
1,1022.0,21.7,18.9,17.2,15.6,81,83,0.6,50.0,15.3,1,2
2,1019.7,20.3,19.3,18.0,18.4,95,91,0.0,40.0,14.2,1,3
3,1018.9,22.3,20.6,19.1,18.8,90,88,1.0,50.0,16.9,1,4
4,1015.9,21.3,20.7,20.2,19.9,95,81,0.0,40.0,13.7,1,5


In [180]:
# transform original data just like train data

def transform_circular(data, column):
    new_data = data.copy()
    new_data[column + '_sin'] = np.sin(2 * np.pi * new_data[column] / max(new_data[column]))
    new_data[column + '_cos'] = np.cos(2 * np.pi * new_data[column] / max(new_data[column]))
    return new_data

df_original = transform_circular(df_original, 'day')
df_original = transform_circular(df_original, 'winddirection')

df_original.drop(['day', 'winddirection'], axis=1, inplace=True)

In [181]:
def generate_combinations(df, exclude_columns=[]):
    selected_columns = [col for col in df.columns if col not in exclude_columns]

    new_columns = {}

    for n in range(2, 3):
        for cols in itertools.combinations(selected_columns, n):
            col_combination = '_'.join(cols)

            # Addition
            new_columns[f'{col_combination}_plus'] = df[list(cols)].sum(axis=1)

            # Subtraction
            if n == 2:
                new_columns[f'{cols[0]}_minus_{cols[1]}'] = df[cols[0]] - df[cols[1]]
                new_columns[f'{cols[1]}_minus_{cols[0]}'] = df[cols[1]] - df[cols[0]]
            else:
                for pair in itertools.combinations(cols, 2):
                    new_columns[f'{pair[0]}_minus_{pair[1]}'] = df[pair[0]] - df[pair[1]]
                    new_columns[f'{pair[1]}_minus_{pair[0]}'] = df[pair[1]] - df[pair[0]]

            # Multiplication
            new_columns[f'{col_combination}_times'] = df[list(cols)].prod(axis=1)

            # Division
            if n == 2:
                new_columns[f'{cols[0]}_div_{cols[1]}'] = df[cols[0]] / df[cols[1]]
                new_columns[f'{cols[1]}_div_{cols[0]}'] = df[cols[1]] / df[cols[0]]
            else:
                for pair in itertools.combinations(cols, 2):
                    new_columns[f'{pair[0]}_div_{pair[1]}'] = df[pair[0]] / df[pair[1]]
                    new_columns[f'{pair[1]}_div_{pair[0]}'] = df[pair[1]] / df[pair[0]]

    df_new = pd.DataFrame(new_columns, index=df.index)

    df_comb = pd.concat([df, df_new], axis=1)
    df_comb.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_comb.fillna(0, inplace=True)
    return df_comb

df_original = generate_combinations(df_original, exclude_columns=['rainfall'])

In [182]:
def add_prev_days(df):

    df_prev = df.copy()

    columns_to_drop = [col for col in df.columns if 'day' in col]

    df_shifted1 = df_prev.shift(1)
    df_shifted1.drop(columns_to_drop, axis=1, inplace=True)

    df_prev = pd.concat([df_prev, df_shifted1.add_prefix("prev1_")], axis=1)

    columns_to_drop = [col for col in df_shifted1.columns if any(keyword in col for keyword in ['winddirection', 'rainfall'])]

    df_shifted1.drop(columns_to_drop, axis=1, inplace=True)

    columns_to_drop = [col for col in df.columns if any(keyword in col for keyword in ['day', 'winddirection', 'rainfall'])]

    df_shifted2 = df.shift(2)
    df_shifted2.drop(columns_to_drop, axis=1, inplace=True)

    df_shifted3 = df.shift(3)
    df_shifted3.drop(columns_to_drop, axis=1, inplace=True)

    df_avg2 = pd.concat([df_shifted1, df_shifted2]).groupby(level=0).mean()
    df_avg2[(df_shifted1.isna()) | (df_shifted2.isna())] = np.nan
    df_avg2 = df_avg2.add_prefix("avg2_")
    df_avg2 = df_avg2.round(1)

    df_avg3 = pd.concat([df_shifted1, df_shifted2, df_shifted3]).groupby(level=0).mean()
    df_avg3[(df_shifted1.isna()) | (df_shifted2.isna()) | (df_shifted3.isna())] = np.nan
    df_avg3 = df_avg3.add_prefix("avg3_")
    df_avg3 = df_avg3.round(1)

    df_prev = pd.concat([df_prev, df_avg2, df_avg3], axis=1)
    df_prev = df_prev.dropna().reset_index(drop=True)

    return df_prev

df_original = add_prev_days(df_original)

In [183]:
df_original.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,windspeed,rainfall,...,avg3_windspeed_minus_cloud,avg3_cloud_windspeed_times,avg3_cloud_div_windspeed,avg3_windspeed_div_cloud,avg3_sunshine_windspeed_plus,avg3_sunshine_minus_windspeed,avg3_windspeed_minus_sunshine,avg3_sunshine_windspeed_times,avg3_sunshine_div_windspeed,avg3_windspeed_div_sunshine
0,1018.9,22.3,20.6,19.1,18.8,90,88,1.0,16.9,1,...,-55.7,1283.6,4.6,0.3,21.9,-15.3,15.3,84.6,0.1,9.4
1,1015.9,21.3,20.7,20.2,19.9,95,81,0.0,13.7,1,...,-71.9,1349.8,5.7,0.2,16.0,-14.9,14.9,8.7,0.0,14.1
2,1018.8,24.3,20.9,19.2,18.0,84,51,7.7,14.5,1,...,-71.7,1296.4,5.8,0.2,15.3,-14.6,14.6,5.6,0.0,5.6
3,1021.8,21.4,18.8,17.0,15.0,79,56,3.4,21.5,0,...,-58.3,1112.1,4.9,0.2,17.9,-12.1,12.1,42.8,0.2,6.3
4,1020.8,21.0,18.4,16.5,14.4,78,28,7.7,14.3,0,...,-46.1,1017.7,4.0,0.3,20.3,-12.9,12.9,61.6,0.2,2.7


In [112]:
df_train.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,windspeed,rainfall,...,avg3_windspeed_minus_cloud,avg3_cloud_windspeed_times,avg3_cloud_div_windspeed,avg3_windspeed_div_cloud,avg3_sunshine_windspeed_plus,avg3_sunshine_minus_windspeed,avg3_windspeed_minus_sunshine,avg3_sunshine_windspeed_times,avg3_sunshine_div_windspeed,avg3_windspeed_div_sunshine
0,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,35.6,1,...,-56.3,1452.4,4.0,0.3,22.2,-15.9,15.9,56.4,0.2,5.9
1,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,24.8,0,...,-52.5,2075.2,3.1,0.3,28.0,-22.4,22.4,50.1,0.2,0.7
2,1022.7,20.6,18.6,16.5,12.5,79.0,81.0,0.0,15.7,1,...,-36.2,1782.9,2.4,0.4,30.1,-22.2,22.2,79.8,0.2,3.0
3,1022.8,19.5,18.4,15.3,11.3,56.0,46.0,7.6,28.4,0,...,-48.3,1923.2,3.2,0.4,26.6,-24.2,24.2,29.8,0.0,2.3
4,1019.7,15.8,13.6,12.7,11.8,96.0,100.0,0.0,52.8,1,...,-34.4,1231.4,2.9,0.5,26.7,-19.2,19.2,101.7,0.1,3.5


In [113]:
df_train.columns = df_train.columns.str.replace('repaired_', '')
print(df_original.columns == df_train.columns)
df_original.columns[~df_original.columns.isin(df_train.columns)]

[ True  True  True ...  True  True  True]


Index([], dtype='object')

In [114]:
# concatenate original and train data

df_train_original = pd.concat([df_original, df_train], axis=0)
df_train_original.reset_index(drop=True, inplace=True)
df_train_original.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,windspeed,rainfall,...,avg3_windspeed_minus_cloud,avg3_cloud_windspeed_times,avg3_cloud_div_windspeed,avg3_windspeed_div_cloud,avg3_sunshine_windspeed_plus,avg3_sunshine_minus_windspeed,avg3_windspeed_minus_sunshine,avg3_sunshine_windspeed_times,avg3_sunshine_div_windspeed,avg3_windspeed_div_sunshine
0,1018.9,22.3,20.6,19.1,18.8,90.0,88.0,1.0,16.9,1,...,-55.7,1283.6,4.6,0.3,21.9,-15.3,15.3,84.6,0.1,9.4
1,1015.9,21.3,20.7,20.2,19.9,95.0,81.0,0.0,13.7,1,...,-71.9,1349.8,5.7,0.2,16.0,-14.9,14.9,8.7,0.0,14.1
2,1018.8,24.3,20.9,19.2,18.0,84.0,51.0,7.7,14.5,1,...,-71.7,1296.4,5.8,0.2,15.3,-14.6,14.6,5.6,0.0,5.6
3,1021.8,21.4,18.8,17.0,15.0,79.0,56.0,3.4,21.5,0,...,-58.3,1112.1,4.9,0.2,17.9,-12.1,12.1,42.8,0.2,6.3
4,1020.8,21.0,18.4,16.5,14.4,78.0,28.0,7.7,14.3,0,...,-46.1,1017.7,4.0,0.3,20.3,-12.9,12.9,61.6,0.2,2.7


In [145]:
X_T = df_train.drop('rainfall', axis=1)
Y_T = df_train['rainfall']
X_O = df_original.drop('rainfall', axis=1)
Y_O = df_original['rainfall']
X_TO = pd.concat([X_T, X_O], axis=0)
Y_TO = pd.concat([Y_T, Y_O], axis=0)

In [126]:
X_T_train, X_T_valid, Y_T_train, Y_T_valid = train_test_split(df_train.drop('rainfall', axis=1), df_train['rainfall'], test_size=0.2, random_state=42)
X_O_train, X_O_valid, Y_O_train, Y_O_valid = train_test_split(df_original.drop('rainfall', axis=1), df_original['rainfall'], test_size=0.2, random_state=42)

X_TO_train = pd.concat([X_T_train, X_O_train], axis=0)
X_TO_valid = pd.concat([X_T_valid, X_O_valid], axis=0)
Y_TO_train = pd.concat([Y_T_train, Y_O_train], axis=0)
Y_TO_valid = pd.concat([Y_T_valid, Y_O_valid], axis=0)

In [127]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [128]:
rf_TO = RandomForestClassifier(n_estimators=300, random_state=42)
rf_TO.fit(X_TO_train, Y_TO_train)

rf_T = RandomForestClassifier(n_estimators=300, random_state=42)
rf_T.fit(X_T_train, Y_T_train)

rf_O = RandomForestClassifier(n_estimators=300, random_state=42)
rf_O.fit(X_O_train, Y_O_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [187]:
# compare the results
res = np.zeros((3,3))
for model, i in zip([rf_TO, rf_T, rf_O], range(3)):
    for (x_, y_, j) in zip([X_TO_valid, X_T_valid, X_O_valid], [Y_TO_valid, Y_T_valid, Y_O_valid], range(3)):
        res[i, j] = roc_auc_score(y_, model.predict_proba(x_)[:,1])

In [188]:
res_df = pd.DataFrame(res, index=['mod_TO', 'mod_T', 'mod_O'], columns=['TO', 'T', 'O'])
res_df

,TO,T,O
mod_TO,0.885658,0.887681,0.860714
mod_T,0.875896,0.875568,0.863095
mod_O,0.876783,0.890010,0.823413
